In [15]:
"""
Ячейка 1: Template-based генерация (без LLM)
"""

def generate_template(query: str, chunks: list[dict]) -> str:
    """
    Генерация ответа из найденных чанков без LLM
    chunks: [{"text": "...", "similarity": 0.87}, ...]
    """
    if not chunks:
        return "К сожалению, я не нашёл информации по вашему вопросу. Хотите связаться с оператором?"
    
    best_chunk = chunks[0]
    
    if best_chunk["similarity"] >= 0.85:
        # Высокая уверенность — даём ответ напрямую
        return best_chunk["text"]
    elif best_chunk["similarity"] >= 0.70:
        # Средняя уверенность — с оговоркой
        return f"По вашему вопросу нашлась следующая информация:\n\n{best_chunk['text']}"
    else:
        return "К сожалению, я не нашёл точного ответа. Хотите связаться с оператором?"

# Тест
test_chunks = [
    {"text": "Возврат товара возможен в течение 14 дней с момента получения.", "similarity": 0.89},
    {"text": "Товар должен сохранять товарный вид и фабричные ярлыки.", "similarity": 0.82},
]

answer = generate_template("Как вернуть товар?", test_chunks)
print(f"Q: Как вернуть товар?")
print(f"A: {answer}")

Q: Как вернуть товар?
A: Возврат товара возможен в течение 14 дней с момента получения.


In [14]:
"""
Ячейка 2: Генерация через GigaChat API (Сбер)
Регистрация: https://developers.sber.ru/portal/products/gigachat
Бесплатный тариф: есть
"""
import requests
import uuid

# 1. Получение токена (нужен Authorization Key из ЛК)
GIGACHAT_AUTH_KEY = "MDE5Y2ZjYTYtMjRiMS03ZTFlLWJmZDktODI3YjRhOGJmNzA0OjNmZmYzNjVhLTU4YWEtNDE4NS05ZGFmLTk1OThjYjk3NTA3Yw=="  # base64 от client_id:client_secret

def get_gigachat_token(auth_key):
    response = requests.post(
        "https://ngw.devices.sberbank.ru:9443/api/v2/oauth",
        headers={
            "Content-Type": "application/x-www-form-urlencoded",
            "Accept": "application/json",
            "RqUID": str(uuid.uuid4()),
            "Authorization": f"Basic {auth_key}"
        },
        data={"scope": "GIGACHAT_API_PERS"},
        verify=False  # Сбер использует свой CA
    )
    return response.json()["access_token"]

def generate_gigachat(query: str, chunks: list[str], token: str) -> str:
    context = "\n---\n".join(chunks)
    
    response = requests.post(
        "https://gigachat.devices.sberbank.ru/api/v1/chat/completions",
        headers={
            "Content-Type": "application/json",
            "Accept": "application/json",
            "Authorization": f"Bearer {token}"
        },
        json={
            "model": "GigaChat",
            "messages": [
                {
                    "role": "system",
                    "content": (
                        "Ты — помощник службы поддержки интернет-магазина. "
                        "Отвечай ТОЛЬКО на основе предоставленного контекста. "
                        "Если в контексте нет ответа на вопрос — скажи об этом. "
                        "Отвечай кратко и по делу, максимум 3 предложения."
                    )
                },
                {
                    "role": "user",
                    "content": f"Контекст:\n{context}\n\nВопрос клиента: {query}"
                }
            ],
            "temperature": 0.1,
            "max_tokens": 200
        },
        verify=False
    )
    
    return response.json()["choices"][0]["message"]["content"]

# Тест (раскомментируй, если есть ключ)
token = get_gigachat_token(GIGACHAT_AUTH_KEY)
answer = generate_gigachat(
    "Какой монитор лучше выбрать?",
     ["Возврат товара возможен в течение 14 дней...", "Товар должен сохранять товарный вид..."],
     token
)
print(answer)

print("ℹ️ Для теста GigaChat:")
print("  1. Зарегистрируйся на developers.sber.ru")
print("  2. Создай проект и получи ключ API")
print("  3. Раскомментируй код выше")

C:\chat-bot-test\chatbot-ml-explore\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'ngw.devices.sberbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
C:\chat-bot-test\chatbot-ml-explore\venv\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'gigachat.devices.sberbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Ваш вопрос не содержит достаточно информации для рекомендации конкретного монитора. Пожалуйста, уточните ваши требования к монитору (разрешение, частота обновления, размер экрана и т.д.).
ℹ️ Для теста GigaChat:
  1. Зарегистрируйся на developers.sber.ru
  2. Создай проект и получи ключ API
  3. Раскомментируй код выше
